In [91]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.options.display.float_format = '{:f}'.format

In [92]:
weekly_data = pd.read_csv("./Data/weekly_aggregated_data.csv")

In [93]:
weekly_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 5350 entries, 0 to 5349
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   avg_global_supply_usd    3217 non-null   float64
 1   avg_supply_on_chain_usd  3128 non-null   float64
 2   blockchain               5350 non-null   str    
 3   chain_supply_share_pct   3128 non-null   float64
 4   currency                 5350 non-null   str    
 5   fee_per_supply_dollar    2966 non-null   float64
 6   gas_fees_native          5350 non-null   float64
 7   gas_fees_usd             5350 non-null   float64
 8   token_symbol             5350 non-null   str    
 9   transfer_count           5350 non-null   int64  
 10  transfer_volume_usd      5350 non-null   float64
 11  unique_receivers         5350 non-null   int64  
 12  unique_senders           5350 non-null   int64  
 13  velocity                 3128 non-null   float64
 14  week                     5350 non-n

### Calculate the Fee-Per-Supply-Dollar Rate

Calculate it for Phase 2

In [94]:
phase2_data = pd.read_csv("../Phase_2/Data/weekly_aggregated_data.csv")
phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7912\1354020066.py:2: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])


In [95]:
phase2_data = phase2_data.sort_values(by='Week')
filtered_phase2_data = phase2_data.tail(12)

In [96]:
avg_fees = filtered_phase2_data['Stablecoin Sequencer Fees (USD)'].mean()
avg_supply = filtered_phase2_data['Stablecoin Supply (USD)'].mean()

print("Average Fees:", avg_fees)
print("Average Supply:", avg_supply)

Average Fees: 56259.16208916667
Average Supply: 3906930594.25


In [97]:
fee_per_supply_dollar_rate = avg_fees / avg_supply
print("Fee-Per-Supply=Dollar-Rate:", fee_per_supply_dollar_rate)

Fee-Per-Supply=Dollar-Rate: 1.4399836580656367e-05


Calculate it for Non-USD Stablecoins

In [98]:
weekly_data = pd.read_csv("./Data/weekly_aggregated_data.csv")
weekly_data['week'] = pd.to_datetime(weekly_data['week'])

In [99]:
print(weekly_data['week'].min())
print(weekly_data['week'].max())

2022-12-26 00:00:00
2026-04-13 00:00:00


In [100]:
tokens = ["EURe", "ZCHF", "XSGD", "GYEN", "CADC", "VEUR", "EURAU"]

filtered_df = weekly_data[
    (weekly_data['blockchain'] == "arbitrum") &
    (weekly_data['token_symbol'].isin(tokens)) &
    (weekly_data['week'] >= '2026-01-26') &
    (weekly_data['week'] < '2026-04-19')
]

In [101]:
filtered_df['week'].unique()

<DatetimeArray>
['2026-04-13 00:00:00', '2026-04-06 00:00:00', '2026-03-30 00:00:00',
 '2026-03-23 00:00:00', '2026-03-16 00:00:00', '2026-03-09 00:00:00',
 '2026-03-02 00:00:00', '2026-02-23 00:00:00', '2026-02-16 00:00:00',
 '2026-02-09 00:00:00', '2026-02-02 00:00:00', '2026-01-26 00:00:00']
Length: 12, dtype: datetime64[us]

In [102]:
filtered_df.groupby('token_symbol')['week'].nunique()

token_symbol
CADC     1
EURe    12
GYEN    10
VEUR     3
XSGD    12
ZCHF    11
Name: week, dtype: int64

In [103]:
token_avg_rates = (
    filtered_df
    .groupby('token_symbol')['fee_per_supply_dollar']
    .mean()
    .reset_index()
    .rename(columns={'fee_per_supply_dollar': 'avg_fee_per_supply_dollar'})
)

In [104]:
token_avg_rates

,token_symbol,avg_fee_per_supply_dollar
0,CADC,0.000000
1,EURe,0.000034
2,GYEN,0.000000
3,VEUR,0.000440
4,XSGD,0.000002
5,ZCHF,0.000001


In [105]:
non_usd_sum = token_avg_rates['avg_fee_per_supply_dollar'].sum()
non_usd_avg_rate = non_usd_sum / 6

In [106]:
print("Per Token Average Rates:")
print(round(token_avg_rates, 10))

print("\nOverall Non-USD Average Rate:")
print(round(non_usd_avg_rate, 10))

Per Token Average Rates:
  token_symbol  avg_fee_per_supply_dollar
0         CADC                   0.000000
1         EURe                   0.000034
2         GYEN                   0.000000
3         VEUR                   0.000440
4         XSGD                   0.000002
5         ZCHF                   0.000001

Overall Non-USD Average Rate:
7.95404e-05


### Calculate the Fee Yield

In [107]:
phase2_data = pd.read_csv("../Phase_2/Data/weekly_aggregated_data.csv")

In [108]:
phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])
phase2_data = phase2_data.sort_values(by='Week')

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7912\1115771419.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])


In [109]:
filtered_phase2_data = phase2_data.tail(12)
print(filtered_phase2_data['Week'].min())
print(filtered_phase2_data['Week'].max())

2026-01-26 00:00:00
2026-04-13 00:00:00


In [110]:
avg_weekly_fees = filtered_phase2_data['Stablecoin Sequencer Fees (USD)'].mean()
avg_weekly_volume = filtered_phase2_data['Stablecoin Transfer Volume (USD)'].mean()
fee_yield = avg_weekly_fees / avg_weekly_volume

In [111]:
print("Average Weekly Fees:", round(avg_weekly_fees, 2))
print("Average Weekly Volume:", round(avg_weekly_volume, 2))
print("Fee Yield:", fee_yield)

Average Weekly Fees: 56259.16
Average Weekly Volume: 16920418464.5
Fee Yield: 3.3249273478195348e-06


### Revenue Projections

In [112]:
weekly_data = pd.read_csv("./Data/weekly_aggregated_data.csv")
weekly_data['week'] = pd.to_datetime(weekly_data['week'])

In [113]:
start_date = "2026-01-26"
end_date = "2026-04-19"

filtered_weekly_data = weekly_data[
    (weekly_data['week'] >= start_date) &
    (weekly_data['week'] <= end_date)
]

In [114]:
filtered_weekly_data.info()

<class 'pandas.DataFrame'>
RangeIndex: 556 entries, 0 to 555
Data columns (total 15 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   avg_global_supply_usd    528 non-null    float64       
 1   avg_supply_on_chain_usd  528 non-null    float64       
 2   blockchain               556 non-null    str           
 3   chain_supply_share_pct   528 non-null    float64       
 4   currency                 556 non-null    str           
 5   fee_per_supply_dollar    528 non-null    float64       
 6   gas_fees_native          556 non-null    float64       
 7   gas_fees_usd             556 non-null    float64       
 8   token_symbol             556 non-null    str           
 9   transfer_count           556 non-null    int64         
 10  transfer_volume_usd      556 non-null    float64       
 11  unique_receivers         556 non-null    int64         
 12  unique_senders           556 non-null    int64 

In [115]:
avg_supply = (
    filtered_weekly_data
    .groupby(['token_symbol', 'week'])['avg_global_supply_usd']
    .mean()
    .reset_index()
    .groupby('token_symbol')['avg_global_supply_usd']
    .mean()
    .reset_index()
    .rename(columns={'avg_global_supply_usd': 'global_supply_usd'})
)

avg_supply

,token_symbol,global_supply_usd
0,AUDD,14458464.833333
1,BRLA,79356320.869048
2,BRZ,1160686812.166667
3,CADC,2240198.666667
4,EURAU,1002700.428571
5,EURC,368052891.666667
6,EURCV,62747962.916667
7,EURI,46980833.023810
8,EURe,24997441.440476
9,GYEN,1058770005.309524


In [116]:
currency_map = (
    filtered_weekly_data[['token_symbol', 'currency']]
    .drop_duplicates()
)

avg_supply = avg_supply.merge(
    currency_map,
    on='token_symbol',
    how='left'
)

avg_supply

,token_symbol,global_supply_usd,currency
0,AUDD,14458464.833333,AUD
1,BRLA,79356320.869048,BRL
2,BRZ,1160686812.166667,BRL
3,CADC,2240198.666667,CAD
4,EURAU,1002700.428571,EUR
5,EURC,368052891.666667,EUR
6,EURCV,62747962.916667,EUR
7,EURI,46980833.023810,EUR
8,EURe,24997441.440476,EUR
9,GYEN,1058770005.309524,JPY


In [117]:
avg_velocity = (
    filtered_weekly_data
    .groupby(['token_symbol', 'week'])['velocity']
    .mean()
    .reset_index()
    .groupby('token_symbol')['velocity']
    .mean()
    .reset_index()
)

avg_velocity

,token_symbol,velocity
0,AUDD,0.371255
1,BRLA,0.880717
2,BRZ,0.097104
3,CADC,2.394030
4,EURAU,0.401413
5,EURC,43.590918
6,EURCV,1.580343
7,EURI,0.200288
8,EURe,4.935011
9,GYEN,0.000304


In [118]:
token_inputs = pd.merge(
    avg_supply,
    avg_velocity,
    on='token_symbol',
    how='inner'
)

token_inputs

,token_symbol,global_supply_usd,currency,velocity
0,AUDD,14458464.833333,AUD,0.371255
1,BRLA,79356320.869048,BRL,0.880717
2,BRZ,1160686812.166667,BRL,0.097104
3,CADC,2240198.666667,CAD,2.394030
4,EURAU,1002700.428571,EUR,0.401413
5,EURC,368052891.666667,EUR,43.590918
6,EURCV,62747962.916667,EUR,1.580343
7,EURI,46980833.023810,EUR,0.200288
8,EURe,24997441.440476,EUR,4.935011
9,GYEN,1058770005.309524,JPY,0.000304


In [119]:
scenarios = {
    "Conservative": 0.10,
    "Base": 0.25,
    "Optimistic": 0.50
}

results = []

for _, row in token_inputs.iterrows():
    token = row['token_symbol']
    currency = row['currency']
    supply = row['global_supply_usd']
    velocity = row['velocity']
    
    for scenario_name, pct in scenarios.items():
        projected_supply = supply * pct
        projected_volume = projected_supply * velocity
        
        results.append({
            "token_symbol": token,
            "currency": currency,
            "scenario": scenario_name,
            "global_supply_usd": supply,
            "velocity": velocity,
            "projected_supply": projected_supply,
            "projected_volume": projected_volume
        })

projection_df = pd.DataFrame(results)

In [120]:
projection_df['global_supply_usd'] = round(projection_df['global_supply_usd'], 2)
projection_df['velocity'] = round(projection_df['velocity'], 2)
projection_df['projected_supply'] = round(projection_df['projected_supply'], 2)
projection_df['projected_volume'] = round(projection_df['projected_volume'], 2)

In [121]:
projection_df.head()

,token_symbol,currency,scenario,global_supply_usd,velocity,projected_supply,projected_volume
0,AUDD,AUD,Conservative,14458464.830000,0.370000,1445846.480000,536778.020000
1,AUDD,AUD,Base,14458464.830000,0.370000,3614616.210000,1341945.050000
2,AUDD,AUD,Optimistic,14458464.830000,0.370000,7229232.420000,2683890.100000
3,BRLA,BRL,Conservative,79356320.870000,0.880000,7935632.090000,6989047.450000
4,BRLA,BRL,Base,79356320.870000,0.880000,19839080.220000,17472618.630000


### Method 1: Supply Based Projection

In [122]:
projection_rate = non_usd_avg_rate
print(projection_rate)

7.954040839908561e-05


In [123]:
projection_df['projected_weekly_fees_supply_model'] = round(
    projection_df['projected_supply'] * projection_rate, 2
)

In [124]:
projection_df['projected_annual_fees_supply_model'] = round(
    projection_df['projected_weekly_fees_supply_model'] * 52, 2
)

In [125]:
projection_df.head()

,token_symbol,currency,scenario,global_supply_usd,velocity,projected_supply,projected_volume,projected_weekly_fees_supply_model,projected_annual_fees_supply_model
0,AUDD,AUD,Conservative,14458464.830000,0.370000,1445846.480000,536778.020000,115.000000,5980.000000
1,AUDD,AUD,Base,14458464.830000,0.370000,3614616.210000,1341945.050000,287.510000,14950.520000
2,AUDD,AUD,Optimistic,14458464.830000,0.370000,7229232.420000,2683890.100000,575.020000,29901.040000
3,BRLA,BRL,Conservative,79356320.870000,0.880000,7935632.090000,6989047.450000,631.200000,32822.400000
4,BRLA,BRL,Base,79356320.870000,0.880000,19839080.220000,17472618.630000,1578.010000,82056.520000


### Method 2: Volume Based Projection

In [126]:
fee_yield = fee_yield
print(fee_yield)

3.3249273478195348e-06


In [127]:
projection_df['projected_weekly_fees_volume_model'] = round(
    projection_df['projected_volume'] * fee_yield, 2
)

In [128]:
projection_df['projected_annual_fees_volume_model'] = round(
    projection_df['projected_weekly_fees_volume_model'] * 52, 2
)

In [129]:
projection_df.head()

,token_symbol,currency,scenario,global_supply_usd,velocity,projected_supply,projected_volume,projected_weekly_fees_supply_model,projected_annual_fees_supply_model,projected_weekly_fees_volume_model,projected_annual_fees_volume_model
0,AUDD,AUD,Conservative,14458464.830000,0.370000,1445846.480000,536778.020000,115.000000,5980.000000,1.780000,92.560000
1,AUDD,AUD,Base,14458464.830000,0.370000,3614616.210000,1341945.050000,287.510000,14950.520000,4.460000,231.920000
2,AUDD,AUD,Optimistic,14458464.830000,0.370000,7229232.420000,2683890.100000,575.020000,29901.040000,8.920000,463.840000
3,BRLA,BRL,Conservative,79356320.870000,0.880000,7935632.090000,6989047.450000,631.200000,32822.400000,23.240000,1208.480000
4,BRLA,BRL,Base,79356320.870000,0.880000,19839080.220000,17472618.630000,1578.010000,82056.520000,58.100000,3021.200000


### Add to Phase 2 Baseline

In [130]:
phase2_data = pd.read_csv("../Phase_2/Data/weekly_aggregated_data.csv")
phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_7912\1354020066.py:2: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  phase2_data['Week'] = pd.to_datetime(phase2_data['Week'])


In [131]:
phase2_data = phase2_data.sort_values(by='Week')

In [132]:
filtered_phase2_data = phase2_data.tail(12)

In [133]:
phase2_total_fees = filtered_phase2_data['Total Sequencer Fees (USD)'].mean()
phase2_total_fees

np.float64(126694.92660833332)

In [134]:
phase2_stablecoin_fees = filtered_phase2_data['Stablecoin Sequencer Fees (USD)'].mean()
phase2_stablecoin_fees

np.float64(56259.16208916667)

In [135]:
projection_df['new_total_fees_supply_model'] = (
    phase2_total_fees +
    projection_df['projected_weekly_fees_supply_model']
)

projection_df['new_total_fees_volume_model'] = (
    phase2_total_fees +
    projection_df['projected_weekly_fees_volume_model']
)

In [136]:
projection_df['new_stablecoin_share_supply_model'] = (
    (phase2_stablecoin_fees + projection_df['projected_weekly_fees_supply_model']) /
    projection_df['new_total_fees_supply_model']
) * 100

projection_df['new_stablecoin_share_volume_model'] = (
    (phase2_stablecoin_fees + projection_df['projected_weekly_fees_volume_model']) /
    projection_df['new_total_fees_volume_model']
) * 100

#### Aggregate by Currency

Aggregate Weekly Fees

In [137]:
currency_agg = (
    projection_df
    .groupby(['currency', 'scenario'])[
        ['projected_weekly_fees_supply_model',
         'projected_weekly_fees_volume_model']
    ]
    .sum()
    .reset_index()
)

Add Total Row (All Tokens)

In [138]:
total_agg = (
    projection_df
    .groupby(['scenario'])[
        ['projected_weekly_fees_supply_model',
         'projected_weekly_fees_volume_model']
    ]
    .sum()
    .reset_index()
)

total_agg['currency'] = "ALL"

In [139]:
final_agg = pd.concat([currency_agg, total_agg], ignore_index=True)

Compute Final Metrics on Aggregated Data    

In [140]:
# Supply Model
final_agg['new_total_fees_supply_model'] = (
    phase2_total_fees +
    final_agg['projected_weekly_fees_supply_model']
)

final_agg['new_stablecoin_share_supply_model'] = (
    (phase2_stablecoin_fees + final_agg['projected_weekly_fees_supply_model']) /
    final_agg['new_total_fees_supply_model']
) * 100

# Volume Model
final_agg['new_total_fees_volume_model'] = (
    phase2_total_fees +
    final_agg['projected_weekly_fees_volume_model']
)

final_agg['new_stablecoin_share_volume_model'] = (
    (phase2_stablecoin_fees + final_agg['projected_weekly_fees_volume_model']) /
    final_agg['new_total_fees_volume_model']
) * 100

In [141]:
final_agg[['currency', 'scenario', 'projected_weekly_fees_supply_model', 'new_total_fees_supply_model', 'new_stablecoin_share_supply_model']].head(6)

,currency,scenario,projected_weekly_fees_supply_model,new_total_fees_supply_model,new_stablecoin_share_supply_model
0,AUD,Base,287.510000,126982.436608,44.531097
1,AUD,Conservative,115.000000,126809.926608,44.455638
2,AUD,Optimistic,575.020000,127269.946608,44.656404
3,BRL,Base,24658.390000,151353.316608,53.462688
4,BRL,Conservative,9863.350000,136558.276608,48.420728
5,BRL,Optimistic,49316.770000,176011.696608,59.982339


In [142]:
final_agg[['currency', 'scenario', 'projected_weekly_fees_volume_model', 'new_total_fees_volume_model', 'new_stablecoin_share_volume_model']].head(6)

,currency,scenario,projected_weekly_fees_volume_model,new_total_fees_volume_model,new_stablecoin_share_volume_model
0,AUD,Base,4.460000,126699.386608,44.407178
1,AUD,Conservative,1.780000,126696.706608,44.406002
2,AUD,Optimistic,8.920000,126703.846608,44.409135
3,BRL,Base,151.790000,126846.716608,44.471748
4,BRL,Conservative,60.710000,126755.636608,44.431848
5,BRL,Optimistic,303.560000,126998.486608,44.538107


### Currency Level Totals

Aggregate at Currency Level

In [143]:
currency_totals = (
    projection_df
    .groupby(['currency', 'scenario'])[
        ['projected_annual_fees_supply_model',
         'projected_annual_fees_volume_model']
    ]
    .sum()
    .reset_index()
)

Add Total Row (All 16 Tokens)

In [144]:
total_row = (
    projection_df
    .groupby('scenario')[
        ['projected_annual_fees_supply_model',
         'projected_annual_fees_volume_model']
    ]
    .sum()
    .reset_index()
)

total_row['currency'] = 'TOTAL'

Combine Currency + Total

In [145]:
final_currency_table = pd.concat(
    [currency_totals, total_row],
    ignore_index=True
)

Pivot for Clean Table Format

In [146]:
pivot_table_supply = final_currency_table.pivot(
    index='currency',
    columns='scenario',
    values='projected_annual_fees_supply_model'
)

pivot_table_supply

scenario,Base,Conservative,Optimistic
currency,,,
AUD,14950.520000,5980.000000,29901.040000
BRL,1282236.280000,512894.200000,2564472.040000
CAD,2316.600000,926.640000,4632.680000
CHF,31362.760000,12545.000000,62725.000000
EUR,522449.720000,208980.200000,1044898.920000
JPY,3318448.120000,1327379.560000,6636896.760000
SGD,19873.360000,7949.240000,39746.720000
TOTAL,5540403.440000,2216161.480000,11080805.840000
TRY,348766.080000,139506.640000,697532.680000


In [147]:
pivot_table_volume = final_currency_table.pivot(
    index='currency',
    columns='scenario',
    values='projected_annual_fees_volume_model'
)

pivot_table_volume

scenario,Base,Conservative,Optimistic
currency,,,
AUD,231.920000,92.560000,463.840000
BRL,7893.080000,3156.920000,15785.120000
CAD,231.920000,92.560000,463.840000
CHF,4326.400000,1731.080000,8653.840000
EUR,703576.120000,281430.240000,1407152.760000
JPY,272.480000,109.200000,544.440000
SGD,1065.480000,426.400000,2130.960000
TOTAL,718364.920000,287345.760000,1436729.840000
TRY,767.520000,306.800000,1535.040000


#### Tokens for Each Currency

In [148]:
projection_df[projection_df['currency'] == 'AUD']['token_symbol'].unique()

<StringArray>
['AUDD']
Length: 1, dtype: str

In [149]:
projection_df[projection_df['currency'] == 'BRL']['token_symbol'].unique()

<StringArray>
['BRLA', 'BRZ']
Length: 2, dtype: str

In [150]:
projection_df[projection_df['currency'] == 'CAD']['token_symbol'].unique()

<StringArray>
['CADC']
Length: 1, dtype: str

In [151]:
projection_df[projection_df['currency'] == 'CHF']['token_symbol'].unique()

<StringArray>
['VCHF', 'ZCHF']
Length: 2, dtype: str

In [152]:
projection_df[projection_df['currency'] == 'EUR']['token_symbol'].unique()

<StringArray>
['EURAU', 'EURC', 'EURCV', 'EURI', 'EURe', 'VEUR']
Length: 6, dtype: str

In [153]:
projection_df[projection_df['currency'] == 'JPY']['token_symbol'].unique()

<StringArray>
['GYEN', 'JPYC']
Length: 2, dtype: str

### Growth Trend Analysis

In [154]:
# LOAD DATA
df = pd.read_csv("./Data/weekly_aggregated_data.csv")
df['week'] = pd.to_datetime(df['week'])

# AGGREGATE (ALL CHAINS → TOKEN LEVEL)
agg = df.groupby(['token_symbol', 'week']).agg(
    supply=('avg_global_supply_usd', 'mean'),
    volume=('transfer_volume_usd', 'sum'),
    senders=('unique_senders', 'sum')
).reset_index()

# YEAR-OVER-YEAR (YoY)

# Define windows
curr_start, curr_end = "2026-01-26", "2026-04-19"
prev_start, prev_end = "2025-01-26", "2025-04-19"

def window_avg(df, token, start, end, col):
    return df[
        (df['token_symbol'] == token) &
        (df['week'] >= start) &
        (df['week'] <= end)
    ][col].mean()

tokens = agg['token_symbol'].unique()

yoy_results = []

for tok in tokens:
    row = {'Token': tok}
    
    for col in ['supply', 'volume', 'senders']:
        curr = window_avg(agg, tok, curr_start, curr_end, col)
        prev = window_avg(agg, tok, prev_start, prev_end, col)
        
        yoy = ((curr - prev) / prev * 100) if prev and not np.isnan(prev) else np.nan
        
        row[f'{col}_yoy_pct'] = yoy
    
    yoy_results.append(row)

yoy_df = pd.DataFrame(yoy_results)

# QUARTER-OVER-QUARTER (QoQ)

# Define quarters
quarters = {
    "2025Q2": ("2025-04-01", "2025-06-30"),
    "2025Q3": ("2025-07-01", "2025-09-30"),
    "2025Q4": ("2025-10-01", "2025-12-31"),
    "2026Q1": ("2026-01-01", "2026-03-31")
}

qoq_results = []

for tok in tokens:
    row = {'Token': tok}
    
    values = {col: [] for col in ['supply', 'volume', 'senders']}
    
    # Calculate quarterly averages
    for q, (start, end) in quarters.items():
        for col in ['supply', 'volume', 'senders']:
            val = window_avg(agg, tok, start, end, col)
            row[f'{col}_{q}'] = val
            values[col].append(val)
    
    # Calculate QoQ
    for col in ['supply', 'volume', 'senders']:
        for i in range(1, len(quarters)):
            prev = values[col][i-1]
            curr = values[col][i]
            
            qoq = ((curr - prev) / prev * 100) if prev and not np.isnan(prev) else np.nan
            
            row[f'{col}_qoq_{i}'] = qoq
    
    qoq_results.append(row)

qoq_df = pd.DataFrame(qoq_results)

# CURRENCY LEVEL AGGREGATION
token_currency = df[['token_symbol', 'currency']].drop_duplicates()

agg = agg.merge(token_currency, on='token_symbol', how='left')

currency_df = df.groupby(['currency', 'week']).agg(
    supply=('avg_global_supply_usd', 'sum'),
    volume=('transfer_volume_usd', 'sum'),
    senders=('unique_senders', 'sum')
).reset_index()

currency_yoy = []

currencies = currency_df['currency'].unique()

for cur in currencies:
    row = {'currency': cur}
    
    for col in ['supply', 'volume', 'senders']:
        curr = currency_df[
            (currency_df['currency'] == cur) &
            (currency_df['week'] >= curr_start) &
            (currency_df['week'] <= curr_end)
        ][col].mean()
        
        prev = currency_df[
            (currency_df['currency'] == cur) &
            (currency_df['week'] >= prev_start) &
            (currency_df['week'] <= prev_end)
        ][col].mean()
        
        yoy = ((curr - prev) / prev * 100) if prev and not np.isnan(prev) else np.nan
        
        row[f'{col}_yoy_pct'] = yoy
    
    currency_yoy.append(row)

currency_yoy_df = pd.DataFrame(currency_yoy)

Token-Level YoY

In [155]:
yoy_df

,Token,supply_yoy_pct,volume_yoy_pct,senders_yoy_pct
0,AUDD,3498.474480,7318.763239,1412.857143
1,BRLA,NaN,66.611444,-12.949478
2,BRZ,331.699102,-2.277420,144.467005
3,CADC,77.119660,118.453336,390.735695
4,EURAU,NaN,NaN,NaN
5,EURC,192.781090,-22.594346,247.711171
6,EURCV,116.988489,13059.857597,13908.860759
7,EURI,NaN,-33.965712,177.548544
8,EURe,18.137449,-39.127011,87.067664
9,GYEN,-29.358624,96.573754,-9.709962


Token-Level QoQ

In [156]:
qoq_df.columns

Index(['Token', 'supply_2025Q2', 'volume_2025Q2', 'senders_2025Q2',
       'supply_2025Q3', 'volume_2025Q3', 'senders_2025Q3', 'supply_2025Q4',
       'volume_2025Q4', 'senders_2025Q4', 'supply_2026Q1', 'volume_2026Q1',
       'senders_2026Q1', 'supply_qoq_1', 'supply_qoq_2', 'supply_qoq_3',
       'volume_qoq_1', 'volume_qoq_2', 'volume_qoq_3', 'senders_qoq_1',
       'senders_qoq_2', 'senders_qoq_3'],
      dtype='str')

##### Supply QoQ

In [157]:
supply_qoq = qoq_df[['Token', 'supply_2025Q2', 
                     'supply_2025Q3', 'supply_qoq_1', 
                     'supply_2025Q4', 'supply_qoq_2',
                     'supply_2026Q1', 'supply_qoq_3']]

supply_qoq

,Token,supply_2025Q2,supply_2025Q3,supply_qoq_1,supply_2025Q4,supply_qoq_2,supply_2026Q1,supply_qoq_3
0,AUDD,467070.747253,4369372.934066,835.484177,11094037.197802,153.904562,13886016.670330,25.166487
1,BRLA,NaN,NaN,NaN,NaN,NaN,79783682.952381,NaN
2,BRZ,280665302.692308,287826339.978022,2.551451,304440645.637363,5.772337,970249911.747253,218.699203
3,CADC,1610455.076923,1785674.857143,10.880141,1585564.967033,-11.206401,2236169.362637,41.032970
4,EURAU,NaN,NaN,NaN,14216418.942857,NaN,989696.109890,-93.038359
5,EURC,186182002.175824,194959597.131868,4.714524,262618470.417582,34.704049,354597673.505494,35.023890
6,EURCV,29377065.571429,44587480.648352,51.776496,53543010.318681,20.085301,59297651.000000,10.747697
7,EURI,NaN,NaN,NaN,45925055.448980,NaN,45265636.296703,-1.435859
8,EURe,22889653.747253,21887094.241758,-4.379968,22711152.505495,3.765042,23831220.549451,4.931797
9,GYEN,1456258454.208791,1210045262.395604,-16.907245,1079458925.153846,-10.791856,1062661695.054945,-1.556079


##### Volume QoQ

In [158]:
volume_qoq = qoq_df[['Token', 'volume_2025Q2', 
                     'volume_2025Q3', 'volume_qoq_1', 
                     'volume_2025Q4', 'volume_qoq_2',
                     'volume_2026Q1', 'volume_qoq_3']]

volume_qoq

,Token,volume_2025Q2,volume_2025Q3,volume_qoq_1,volume_2025Q4,volume_qoq_2,volume_2026Q1,volume_qoq_3
0,AUDD,94234.641142,1341494.656444,1323.568488,9135615.713685,581.002766,2541217.522343,-72.183402
1,BRLA,11778883.243598,12829234.844555,8.917243,16621915.901030,29.562800,26843636.424261,61.495441
2,BRZ,28873819.668819,4453064.584551,-84.577501,16067677.944918,260.822926,29914309.054010,86.176927
3,CADC,9338192.432149,14538763.609860,55.691412,4774793.792283,-67.158185,7848679.835920,64.377357
4,EURAU,NaN,12284427.010990,NaN,15013633.164691,22.216797,204923.379969,-98.635085
5,EURC,2933393563.950093,4614620176.349300,57.313367,2627100258.286329,-43.070065,1727788142.980074,-34.232120
6,EURCV,506961.649940,16254102.721270,3106.179939,40399080.050707,148.546971,70471123.674862,74.437447
7,EURI,15086526.241990,14121017.022342,-6.399811,10472028.720955,-25.840832,9679833.696257,-7.564867
8,EURe,127409077.821324,131340124.614776,3.085374,120090452.022091,-8.565298,109133464.086397,-9.123946
9,GYEN,336848.210426,581187.336711,72.536863,70721.812616,-87.831495,254260.770153,259.522417


##### Senders QoQ

In [159]:
senders_qoq = qoq_df[['Token', 'senders_2025Q2', 
                     'senders_2025Q3', 'senders_qoq_1', 
                     'senders_2025Q4', 'senders_qoq_2',
                     'senders_2026Q1', 'senders_qoq_3']]

senders_qoq

,Token,senders_2025Q2,senders_2025Q3,senders_qoq_1,senders_2025Q4,senders_qoq_2,senders_2026Q1,senders_qoq_3
0,AUDD,19.076923,29.000000,52.016129,197.230769,580.106101,162.923077,-17.394696
1,BRLA,4719.538462,5021.692308,6.402191,3415.000000,-31.995037,3341.000000,-2.166911
2,BRZ,704.615385,843.769231,19.748908,1171.615385,38.854955,1614.153846,37.771650
3,CADC,102.769231,153.769231,49.625749,164.000000,6.653327,264.923077,61.538462
4,EURAU,NaN,7.800000,NaN,33.384615,328.007890,20.769231,-37.788018
5,EURC,3862.230769,7390.923077,91.364098,7617.076923,3.059886,8710.846154,14.359435
6,EURCV,10.076923,19.538462,93.893130,171.769231,779.133858,697.692308,306.180027
7,EURI,98.230769,113.846154,15.896633,153.538462,34.864865,177.538462,15.631263
8,EURe,5751.000000,7702.230769,33.928548,8067.307692,4.739886,8816.230769,9.283433
9,GYEN,74.230769,69.000000,-7.046632,48.384615,-29.877369,58.153846,20.190779


Currency-Level YoY

In [160]:
currency_yoy_df

,currency,supply_yoy_pct,volume_yoy_pct,senders_yoy_pct
0,AUD,6533.170214,7318.763239,1412.857143
1,BRL,1107.625498,20.919545,8.954460
2,CAD,106.912806,118.453336,390.735695
3,CHF,446.578167,647.759999,686.547085
4,EUR,171.952137,-21.997640,154.816630
5,JPY,193.804167,3153.201861,7003.530895
6,SGD,NaN,-81.478425,-40.912801
7,TRY,NaN,-97.137420,-32.605730


### Cross Token Comparison Table

In [161]:
weekly_data = pd.read_csv("./Data/weekly_aggregated_data.csv")
weekly_data['week'] = pd.to_datetime(weekly_data['week'])

In [162]:
weekly_data.columns

Index(['avg_global_supply_usd', 'avg_supply_on_chain_usd', 'blockchain',
       'chain_supply_share_pct', 'currency', 'fee_per_supply_dollar',
       'gas_fees_native', 'gas_fees_usd', 'token_symbol', 'transfer_count',
       'transfer_volume_usd', 'unique_receivers', 'unique_senders', 'velocity',
       'week'],
      dtype='str')

Filter Last 12 Weeks

In [163]:
start_date = "2026-01-26"
end_date   = "2026-04-19"

filtered_data = weekly_data[
    (weekly_data['week'] >= start_date) &
    (weekly_data['week'] <= end_date)
]

Base Aggregations (Token Level)

In [164]:
token_metrics = (
    filtered_data
    .groupby('token_symbol')
    .agg({
        'currency': 'first',
        'transfer_volume_usd': 'sum',
        'gas_fees_usd': 'sum'
    })
    .reset_index()
)

In [165]:
avg_supply = (
    filtered_data
    .groupby(['token_symbol', 'week'])['avg_global_supply_usd']
    .mean()
    .reset_index()
    .groupby('token_symbol')['avg_global_supply_usd']
    .mean()
    .reset_index()
)

token_metrics = token_metrics.merge(
    avg_supply,
    left_on='token_symbol',
    right_on='token_symbol',
    how='left'
)

In [166]:
avg_velocity = (
    filtered_data
    .groupby(['token_symbol', 'week'])['velocity']
    .mean()
    .reset_index()
    .groupby('token_symbol')['velocity']
    .mean()
    .reset_index()
)

token_metrics = token_metrics.merge(
    avg_velocity,
    left_on='token_symbol',
    right_on='token_symbol',
    how='left'
)

In [167]:
avg_fee_per_supply_dollar = (
    filtered_data
    .groupby(['token_symbol', 'week'])['fee_per_supply_dollar']
    .mean()
    .reset_index()
    .groupby('token_symbol')['fee_per_supply_dollar']
    .mean()
    .reset_index()
)

token_metrics = token_metrics.merge(
    avg_fee_per_supply_dollar,
    left_on='token_symbol',
    right_on='token_symbol',
    how='left'
)

In [168]:
token_metrics['weekly_transfer_volume_usd'] = (
    token_metrics['transfer_volume_usd'] / 12
)

token_metrics['weekly_gas_fees_usd'] = (
    token_metrics['gas_fees_usd'] / 12
)

In [169]:
token_metrics = token_metrics.rename(columns={
    'token_symbol': 'Token',
    'currency': 'Currency',
    'avg_global_supply_usd': 'Global Supply (USD)',
    'weekly_transfer_volume_usd': 'Weekly Transfer Volume (USD)',
    'weekly_gas_fees_usd': 'Weekly Gas Fees (USD)',
    'velocity': 'Average Velocity',
    'fee_per_supply_dollar': 'Fee Per Supply Dollar'
})

token_metrics = token_metrics.drop(columns=['transfer_volume_usd', 'gas_fees_usd'])

In [170]:
token_metrics

,Token,Currency,Global Supply (USD),Average Velocity,Fee Per Supply Dollar,Weekly Transfer Volume (USD),Weekly Gas Fees (USD)
0,AUDD,AUD,14458464.833333,0.371255,0.000025,2483573.576735,160.573515
1,BRLA,BRL,79356320.869048,0.880717,0.000003,27114851.123967,167.703532
2,BRZ,BRL,1160686812.166667,0.097104,0.000002,31326030.682641,226.126047
3,CADC,CAD,2240198.666667,2.394030,0.000047,9122111.453230,161.659583
4,EURAU,EUR,1002700.428571,0.401413,0.000018,259328.252119,12.352009
5,EURC,EUR,368052891.666667,43.590918,0.000084,6701238601.295636,20189.623093
6,EURCV,EUR,62747962.916667,1.580343,0.000060,85083041.022733,3254.879648
7,EURI,EUR,46980833.023810,0.200288,0.000007,9707339.989110,389.880644
8,EURe,EUR,24997441.440476,4.935011,0.000034,111540285.782312,505.088611
9,GYEN,JPY,1058770005.309524,0.000304,0.000000,555590.700331,52.835682


Top Chain Calculation

In [171]:
chain_supply = (
    filtered_data
    .groupby(['token_symbol', 'blockchain'])['avg_supply_on_chain_usd']
    .mean()
    .reset_index()
)

top_chain_idx = chain_supply.groupby('token_symbol')['avg_supply_on_chain_usd'].idxmax()

top_chain = chain_supply.loc[top_chain_idx]

In [172]:
top_chain_share = (
    filtered_data
    .groupby(['token_symbol', 'blockchain'])['chain_supply_share_pct']
    .mean()
    .reset_index()
)

top_chain = pd.merge(
    top_chain,
    top_chain_share,
    on=['token_symbol', 'blockchain'],
    how='left'
)

In [173]:
top_chain = top_chain.rename(columns={
    'token_symbol': 'Token',
    'blockchain': 'Top Chain',
    'chain_supply_share_pct': 'Top Chain Supply Share (%)'
})[['Token', 'Top Chain', 'Top Chain Supply Share (%)']]

In [174]:
top_chain

,Token,Top Chain,Top Chain Supply Share (%)
0,AUDD,ethereum,29.873733
1,BRLA,polygon,18.908733
2,BRZ,polygon,5.327738
3,CADC,base,58.116558
4,EURAU,ethereum,68.326341
5,EURC,ethereum,65.816264
6,EURCV,ethereum,83.887761
7,EURI,ethereum,93.659428
8,EURe,gnosis,72.375833
9,GYEN,ethereum,90.298281


On Arbitrum?

In [175]:
arbitrum_tokens = (
    filtered_data[filtered_data['blockchain'] == 'arbitrum']['token_symbol']
    .unique()
)

token_metrics['On Arbitrum?'] = token_metrics['Token'].isin(arbitrum_tokens)
token_metrics['On Arbitrum?'] = token_metrics['On Arbitrum?'].map({True: 'Yes', False: 'No'})

Add YoY Growth

In [176]:
yoy_supply = yoy_df[['Token', 'supply_yoy_pct']]

yoy_supply = yoy_supply.rename(columns={
    'token_symbol': 'Token',
    'supply_yoy_pct': 'YoY Supply Growth (%)'
})

Merge Everything

In [177]:
final_table = token_metrics.merge(top_chain, on='Token', how='left')
final_table = final_table.merge(yoy_supply, on='Token', how='left')

Final Column Order

In [178]:
final_table = final_table[
    [
        'Token',
        'Currency',
        'Global Supply (USD)',
        'Top Chain',
        'Top Chain Supply Share (%)',
        'Weekly Transfer Volume (USD)',
        'Weekly Gas Fees (USD)',
        'Average Velocity',
        'Fee Per Supply Dollar',
        'On Arbitrum?',
        'YoY Supply Growth (%)'
    ]
]

In [179]:
final_table['Global Supply (USD)'] = round(final_table['Global Supply (USD)'], 2)
final_table['Top Chain Supply Share (%)'] = round(final_table['Top Chain Supply Share (%)'], 2)
final_table['Weekly Transfer Volume (USD)'] = round(final_table['Weekly Transfer Volume (USD)'], 2)
final_table['Weekly Gas Fees (USD)'] = round(final_table['Weekly Gas Fees (USD)'], 2)
final_table['Average Velocity'] = round(final_table['Average Velocity'], 2)
final_table['YoY Supply Growth (%)'] = round(final_table['YoY Supply Growth (%)'], 2)

In [180]:
final_table

,Token,Currency,Global Supply (USD),Top Chain,Top Chain Supply Share (%),Weekly Transfer Volume (USD),Weekly Gas Fees (USD),Average Velocity,Fee Per Supply Dollar,On Arbitrum?,YoY Supply Growth (%)
0,AUDD,AUD,14458464.830000,ethereum,29.870000,2483573.580000,160.570000,0.370000,0.000025,No,3498.470000
1,BRLA,BRL,79356320.870000,polygon,18.910000,27114851.120000,167.700000,0.880000,0.000003,No,NaN
2,BRZ,BRL,1160686812.170000,polygon,5.330000,31326030.680000,226.130000,0.100000,0.000002,No,331.700000
3,CADC,CAD,2240198.670000,base,58.120000,9122111.450000,161.660000,2.390000,0.000047,Yes,77.120000
4,EURAU,EUR,1002700.430000,ethereum,68.330000,259328.250000,12.350000,0.400000,0.000018,No,NaN
5,EURC,EUR,368052891.670000,ethereum,65.820000,6701238601.300000,20189.620000,43.590000,0.000084,No,192.780000
6,EURCV,EUR,62747962.920000,ethereum,83.890000,85083041.020000,3254.880000,1.580000,0.000060,No,116.990000
7,EURI,EUR,46980833.020000,ethereum,93.660000,9707339.990000,389.880000,0.200000,0.000007,No,NaN
8,EURe,EUR,24997441.440000,gnosis,72.380000,111540285.780000,505.090000,4.940000,0.000034,Yes,18.140000
9,GYEN,JPY,1058770005.310000,ethereum,90.300000,555590.700000,52.840000,0.000000,0.000000,Yes,-29.360000
